# Notebook 02: KldB Import & Mapping to ESCO

Establishment of a robust mapping bridge from KldB (DE) to ESCO (EU) via ISCO-08. KldB provides a robust German occupational structure but does not include a skills taxonomy; ESCO is skills-oriented and internationally compatible:
- Import KldB 2010 (2020), format into a clean table, 11/14/2025
- Import official conversion key KldB -> ISCO-08
- Create ISCO-08 and ESCO-ISCO group (`ISCOGroups_en.csv`) mapping table `kldb_esco_mapping`: KldB code -> ISCO-08 code -> ESCO occupation (`occupation_uri`); Mapping forms the basis for later transferring ESCO skill relations to KldB occupations (Notebooks 03-04).

(Sources: Bundesagentur für Arbeit, https://statistik.arbeitsagentur.de/DE/Navigation/Grundlagen/Klassifikationen/Klassifikation-der-Berufe/Klassifikation-der-Berufe-Nav.html; https://statistik.arbeitsagentur.de/DE/Navigation/Grundlagen/Klassifikationen/Klassifikation-der-Berufe/KldB2010-Fassung2020/KldB2010-Fassung2020-Nav.html; https://metadaten.bibb.de/de/group/classification/1; https://statistik.arbeitsagentur.de/DE/Statischer-Content/Grundlagen/Klassifikationen/Klassifikation-der-Berufe/KldB2010-Fassung2020/Arbeitsmittel/Umschluesselungstabellen.html; ESCO, https://esco.ec.europa.eu/en/about-esco/escopedia/escopedia/mapping-esco)

## 1. Import and display KLDB raw data

In [1]:
from pathlib import Path # Imports as in 01
import pandas as pd

In [2]:
import sys
print("Python executable im Notebook:", sys.executable)
import subprocess

# Note: openpyxl is required to read KldB Excel files; installation is part of the setup
# Install openpyxl in the interpreter
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "openpyxl"])
subprocess.check_call([sys.executable, "-m", "pip", "show", "openpyxl"]) # Inspection

Python executable im Notebook: C:\Users\sigle\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe


0

In [3]:
# Project Root
PROJECT_ROOT = Path.cwd().resolve().parent

DATA_RAW_KLDB = PROJECT_ROOT / "data" / "raw" / "kldb"
DATA_INTERIM   = PROJECT_ROOT / "data" / "interim"

print("Project root:", PROJECT_ROOT)
print("KldB raw path:", DATA_RAW_KLDB)
print("Interim path:", DATA_INTERIM)

DATA_INTERIM.mkdir(parents=True, exist_ok=True)

Project root: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
KldB raw path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw\kldb
Interim path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim


In [4]:
# Information about KLDB
list(DATA_RAW_KLDB.glob("*"))

[WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/kldb/Alphabetisches-Verzeichnis-Berufsbenennungen.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/kldb/Alphabetisches-Verzeichnis-Berufsbenennungen.xlsx'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/kldb/Kldb2010-ueF2020-Englisch.xlsx'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/kldb/kldb_titles_en.csv'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/kldb/Systematisches-Verzeichnis-KldB-2020.xlsx'),
 WindowsPath('C:/Users/sigle/OneDrive - Hochschule Reutlingen/Dokumente/Profilerweiterung/data/raw/kldb/Umsteigeschluessel-KLDB2020-ISCO08.xlsx')]

In [5]:
# Load KldB
kldb_systematik_path = DATA_RAW_KLDB / "Systematisches-Verzeichnis-KldB-2020.xlsx"
print(kldb_systematik_path)

# Sheet: Short Names
kldb_raw = pd.read_excel(
    kldb_systematik_path,
    sheet_name="Systematik_Kurzbezeichnungen",
    skiprows=4,
    dtype=str
)

print("KldB raw shape:", kldb_raw.shape)
kldb_raw.head()

C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw\kldb\Systematisches-Verzeichnis-KldB-2020.xlsx
KldB raw shape: (2195, 3)


,KldB 2010,Kurzbezeichnungen der Systematikpositionen,Filter zur Auswahl der Ebenen
0,1,"Land-, Forst-, Tierwirtschaft, Gartenbau",KldB2010_1
1,11,"Land-, Tier-, Forstwirtschaftsberufe",KldB2010_2
2,111,Landwirtschaft,KldB2010_3
3,1110,Berufe in der Landwirtschaft (o.S.),KldB2010_4
4,11101,Landwirtschaft (o.S.) - Helfer,KldB2010_5


In [6]:
# View sheets
xls = pd.ExcelFile(kldb_systematik_path)
print("Sheets in der KldB-Datei:", xls.sheet_names)

Sheets in der KldB-Datei: ['Impressum', 'Übersicht', 'Übersicht_Systematikpositionen', 'Systematik_Langbezeichnungen', 'Systematik_Kurzbezeichnungen']


## 2. Processing of the KldB Classification System (5-digit codes)

The KldB Classification System provides codes and designations (short/long) at various levels; here, the data is reduced to 5-digit codes (occupational categories). The short designations are used for this reduction, as they are concise and sufficient for subsequent profile aggregation. Long designations are added below.

In [7]:
# Rename columns
kldb_clean = kldb_raw.rename(columns={"KldB 2010": "kldb_code", "Kurzbezeichnungen der Systematikpositionen": "kldb_title_short"})

# Keep lines with actual code
kldb_clean = kldb_clean.dropna(subset=["kldb_code"])

# Remove whitespace, as a string
kldb_clean["kldb_code"] = kldb_clean["kldb_code"].astype(str).str.strip()
kldb_clean["kldb_title_short"] = kldb_clean["kldb_title_short"].astype(str).str.strip()

# Auxiliary column, code length
kldb_clean["code_len"] = kldb_clean["kldb_code"].str.len()

# Keep only 5-digit codes (occupational categories)
kldb_5 = kldb_clean[kldb_clean["code_len"] == 5].copy()
kldb_5 = kldb_5.drop(columns="code_len")

print("Anzahl 5-stelliger KldB-Codes:", len(kldb_5))
kldb_5.head()

Anzahl 5-stelliger KldB-Codes: 1300


,kldb_code,kldb_title_short,Filter zur Auswahl der Ebenen
4,11101,Landwirtschaft (o.S.) - Helfer,KldB2010_5
5,11102,Landwirtschaft (o.S.) - Fachkraft,KldB2010_5
6,11103,Landwirtschaft (o.S.) - Spezialist,KldB2010_5
7,11104,Landwirtschaft (o.S.) - Experte,KldB2010_5
9,11113,Landtechnik - Spezialist,KldB2010_5


## 3. Add long descriptions for the KldB codes

In [8]:
# Read long descriptions
kldb_long_raw = pd.read_excel(
    kldb_systematik_path,
    sheet_name="Systematik_Langbezeichnungen",
    skiprows=4, # due to meta lines above
    dtype=str
)

kldb_long = kldb_long_raw.rename(columns={
    "KldB 2010": "kldb_code",
    "Langbezeichnungen der Systematikpositionen": "kldb_title_long"
})

kldb_long = kldb_long.dropna(subset=["kldb_code"])
kldb_long["kldb_code"] = kldb_long["kldb_code"].astype(str).str.strip()

# Merge short and long names
kldb_5 = kldb_5.merge(
    kldb_long[["kldb_code", "kldb_title_long"]],
    on="kldb_code",
    how="left"
)

kldb_5.head()

,kldb_code,kldb_title_short,Filter zur Auswahl der Ebenen,kldb_title_long
0,11101,Landwirtschaft (o.S.) - Helfer,KldB2010_5,Berufe in der Landwirtschaft (ohne Spezialisie...
1,11102,Landwirtschaft (o.S.) - Fachkraft,KldB2010_5,Berufe in der Landwirtschaft (ohne Spezialisie...
2,11103,Landwirtschaft (o.S.) - Spezialist,KldB2010_5,Berufe in der Landwirtschaft (ohne Spezialisie...
3,11104,Landwirtschaft (o.S.) - Experte,KldB2010_5,Berufe in der Landwirtschaft (ohne Spezialisie...
4,11113,Landtechnik - Spezialist,KldB2010_5,Berufe in der Landtechnik - komplexe Spezialis...


## 4. Conversion Key: KldB 2010 (5-digit) -> ISCO-08 (4-digit)

The conversion key is the official mapping from KldB 5-digit to ISCO-08, including indicators for unique vs. non-unique mappings. (https://statistik.arbeitsagentur.de/DE/Statischer-Content/Grundlagen/Klassifikationen/Klassifikation-der-Berufe/KldB2010-Fassung2020/Arbeitsmittel/Umschluesselungstabellen.html). 5-digit KldB occupational categories form the basis for future skill profiles.

In [9]:
kldb_isco_path = next(DATA_RAW_KLDB.glob("*ISCO*08*.xlsx"))

# Scan the sheet using the transfer key
kldb_isco_raw = pd.read_excel(
    kldb_isco_path,
    sheet_name="Umsteiger KldB üF 2020 auf ISCO",
    header=4, # Skip meta lines
    dtype=str
)

print("Verwendete Datei:", kldb_isco_path.name)
print("Spalten im Umstiegsschlüssel:")
print(kldb_isco_raw.columns.tolist())

kldb_isco_raw.head(5)

Verwendete Datei: Umsteigeschluessel-KLDB2020-ISCO08.xlsx
Spalten im Umstiegsschlüssel:
['KldB 2010\n(5-Steller)', 'Bezeichnungen der KldB2010 (5-Steller)', 'Classification title (English)', 'ISCO-08\n(4-Steller)', 'Bezeichnungen der ISCO-08 (4-Steller)', 'Unit Group (English)', 'Umstieg eindeutig (1);\nnicht eindeutig (0)', 'Schwerpunkt (1) und \nAnzahl der Alternativen']


,KldB 2010\n(5-Steller),Bezeichnungen der KldB2010 (5-Steller),Classification title (English),ISCO-08\n(4-Steller),Bezeichnungen der ISCO-08 (4-Steller),Unit Group (English),Umstieg eindeutig (1);\nnicht eindeutig (0),Schwerpunkt (1) und \nAnzahl der Alternativen
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9213,Hilfsarbeiter in Ackerbau und Tierhaltung (ohn...,Mixed crop and livestock farm labourers,0,2
2,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6111,Ackerbauern und Gemüseanbauer,Field crop and vegetable growers,0,2
3,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6130,Landwirte mit Ackerbau und Tierhaltung (ohne a...,Mixed crop and animal producers,0,1
4,11103,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,2132,"Agrar-, Forst- und Fischereiwissenschaftler un...","Farming, forestry and fisheries advisers",1,1


Mapping of Conversion Codes: KldB 5-digit -> ISCO-08 4-digit

In [10]:
# Rename columns
kldb_isco = kldb_isco_raw.rename(columns={
    "KldB 2010\n(5-Steller)": "kldb_5_code",
    "Bezeichnungen der KldB2010 (5-Steller)": "kldb_title_de",
    "Classification title (English)": "kldb_title_en",
    "ISCO-08\n(4-Steller)": "isco08_4",
    "Bezeichnungen der ISCO-08 (4-Steller)": "isco_title_de",
    "Unit Group (English)": "isco_title_en",
    "Umstieg eindeutig (1);\nnicht eindeutig (0)": "unambiguous_flag",
    "Schwerpunkt (1) und \nAnzahl der Alternativen": "focus_and_alt_count",
})

# Remove whitespace
for col in ["kldb_5_code", "kldb_title_de", "kldb_title_en","isco08_4", "isco_title_de", "isco_title_en","unambiguous_flag", "focus_and_alt_count"]:
    if col in kldb_isco.columns:
        kldb_isco[col] = kldb_isco[col].fillna("").astype(str).str.strip()

# Keep only rows with 5-digit KldB codes and 4-digit ISCO codes
kldb_isco = kldb_isco[kldb_isco["kldb_5_code"].str.len() == 5]
kldb_isco = kldb_isco[kldb_isco["isco08_4"].str.len() == 4]

print("Anzahl Mapping-Zeilen nach Bereinigung:", len(kldb_isco))
kldb_isco.head(10)

Anzahl Mapping-Zeilen nach Bereinigung: 1523


,kldb_5_code,kldb_title_de,kldb_title_en,isco08_4,isco_title_de,isco_title_en,unambiguous_flag,focus_and_alt_count
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9213,Hilfsarbeiter in Ackerbau und Tierhaltung (ohn...,Mixed crop and livestock farm labourers,0,2
2,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6111,Ackerbauern und Gemüseanbauer,Field crop and vegetable growers,0,2
3,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6130,Landwirte mit Ackerbau und Tierhaltung (ohne a...,Mixed crop and animal producers,0,1
4,11103,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,2132,"Agrar-, Forst- und Fischereiwissenschaftler un...","Farming, forestry and fisheries advisers",1,1
5,11104,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,2132,"Agrar-, Forst- und Fischereiwissenschaftler un...","Farming, forestry and fisheries advisers",1,1
6,11113,Berufe in der Landtechnik - komplexe Spezialis...,Technical occupations in farming-complex tasks,3142,Agrartechniker,Agricultural technicians,1,1
7,11114,Berufe in der Landtechnik - hoch komplexe Täti...,Technical occupations in farming-highly comple...,2132,"Agrar-, Forst- und Fischereiwissenschaftler un...","Farming, forestry and fisheries advisers",1,1
8,11123,Landwirtschaftliche Sachverständige - komplexe...,Agricultural experts-complex tasks,3315,Schätzer und Schadensgutachter,Valuers and loss assessors,1,1
9,11124,Landwirtschaftliche Sachverständige - hoch kom...,Agricultural experts-high complex tasks,2132,"Agrar-, Forst- und Fischereiwissenschaftler un...","Farming, forestry and fisheries advisers",1,1


Check unique vs. non-unique transfers

In [11]:
if "unambiguous_flag" in kldb_isco.columns:
    print(kldb_isco["unambiguous_flag"].value_counts(dropna=False))
else:
    print("Spalte 'unambiguous_flag' nicht vorhanden.")

unambiguous_flag
1    1136
0     387
Name: count, dtype: int64


1,136 cases of unambiguous mapping, 387 cases of ambiguous mapping.

Mapping between KldB and ESCO is described using SKOS relations (`exactMatch`, `closeMatch`, `narrowMatch`, `broadMatch`), (W3C, 2009); these relations indicate how closely two concepts are related in a subject-matter context

In the BA’s conversion key, SKOS types are not explicitly stored; instead, an aggregated rating is used:
- `unambiguous_flag = 1` -> unambiguous conversion (exactly one target code)
- `unambiguous_flag = 0` -> non-unambiguous mapping (multiple target codes)
- `focus_and_alt_count` -> number of alternatives listed in the original key per KldB code

Non-unambiguous mappings result in a KldB code being linked to multiple ISCO codes and thus multiple ESCO occupations.

## 5. Import ISCO groups from ESCO

Mapped from ESCO ISCO-08 codes to an ESCO occupation (`occupation_uri`), table `ISCOGroups_en.csv`

In [12]:
# Path to ESCO raw data (as in Notebook 01)
DATA_RAW_ESCO = PROJECT_ROOT / "data" / "raw" / "esco"
print("ESCO raw path:", DATA_RAW_ESCO)

ESCO raw path: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw\esco


In [13]:
# Path to the ISCO group CSV file
isco_groups_path = DATA_RAW_ESCO / "ISCOGroups_en.csv"
print("ISCOGroups-Datei:", isco_groups_path)

# Comma-separated file
isco_groups_raw = pd.read_csv(isco_groups_path, sep=",", dtype=str)

print("Shape ISCOGroups_en:", isco_groups_raw.shape)
print("Spalten in ISCOGroups_en:")
print(isco_groups_raw.columns.tolist())

isco_groups_raw.head(5)

ISCOGroups-Datei: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\raw\esco\ISCOGroups_en.csv
Shape ISCOGroups_en: (619, 8)
Spalten in ISCOGroups_en:
['conceptType', 'conceptUri', 'code', 'preferredLabel', 'status', 'altLabels', 'inScheme', 'description']


,conceptType,conceptUri,code,preferredLabel,status,altLabels,inScheme,description
0,ISCOGroup,http://data.europa.eu/esco/isco/C0,0,Armed forces occupations,released,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Armed forces occupations include all jobs held...
1,ISCOGroup,http://data.europa.eu/esco/isco/C01,01,Commissioned armed forces officers,released,NaN,http://data.europa.eu/esco/concept-scheme/isco...,Commissioned armed forces officers provide lea...
2,ISCOGroup,http://data.europa.eu/esco/isco/C011,011,Commissioned armed forces officers,released,NaN,http://data.europa.eu/esco/concept-scheme/isco...,Commissioned armed forces officers provide lea...
3,ISCOGroup,http://data.europa.eu/esco/isco/C0110,0110,Commissioned armed forces officers,released,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Commissioned armed forces officers provide lea...
4,ISCOGroup,http://data.europa.eu/esco/isco/C02,02,Non-commissioned armed forces officers,released,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Non-commissioned armed forces officers enforce...


In [14]:
# Keep relevant columns and clean up
isco_groups = pd.DataFrame({
    "isco_code": isco_groups_raw["code"].astype(str).str.strip(),
    "isco_title_en": isco_groups_raw["preferredLabel"].astype(str).str.strip() # ISCO-Codes werden als String behandelt, wegen führender Nullen
})

# Keep only 4-digit ISCO codes
mask_4digit = isco_groups["isco_code"].str.fullmatch(r"\d{4}", na=False)
isco_groups = isco_groups[mask_4digit].copy()

print("Anzahl 4-stelliger ISCO-Gruppen:", len(isco_groups))
isco_groups.head(10)

Anzahl 4-stelliger ISCO-Gruppen: 436


,isco_code,isco_title_en
3,0110,Commissioned armed forces officers
6,0210,Non-commissioned armed forces officers
9,0310,"Armed forces occupations, other ranks"
13,1111,Legislators
14,1112,Senior government officials
15,1113,Traditional chiefs and heads of village
16,1114,Senior officials of special-interest organisat...
18,1120,Managing directors and chief executives
21,1211,Finance managers
22,1212,Human resource managers


## 6. From ISCO to ESCO

ESCO occupations reference ISCO groups via iscoGroup; therefore, the 4-digit ISCO code can be used as a join key to derive ISCO -> ESCO occupations (bridge table).

In [15]:
DATA_RAW_ESCO = PROJECT_ROOT / "data" / "raw" / "esco"

occ_path = DATA_RAW_ESCO / "occupations_en.csv"
occ_raw = pd.read_csv(occ_path, sep=",", dtype=str)  # Comma-separated

print("Spalten occupations_en.csv:")
print(occ_raw.columns.tolist())
occ_raw.head(5)

Spalten occupations_en.csv:
['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code']


,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,technical and operations director\nhead of tec...,NaN,released,2024-01-25T11:28:50.295Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7
1,Occupation,http://data.europa.eu/esco/occupation/000e93a3...,8121,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,NaN,released,2024-01-23T10:09:32.099Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Metal drawing machine operators set up and ope...,8121.4
2,Occupation,http://data.europa.eu/esco/occupation/0019b951...,7543,precision device inspector,inspector of precision instruments\nprecision ...,NaN,released,2024-01-25T15:00:12.188Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Precision device inspectors make sure precisio...,7543.10.3
3,Occupation,http://data.europa.eu/esco/occupation/0022f466...,3155,air traffic safety technician,air traffic safety electronics hardware specia...,NaN,released,2024-01-29T16:01:13.998Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Air traffic safety technicians provide technic...,3155.1
4,Occupation,http://data.europa.eu/esco/occupation/002da35b...,2431,hospitality revenue manager,hospitality revenues manager\nyield manager\nh...,NaN,released,2024-01-11T10:28:45.871Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Hospitality revenue managers maximise revenue ...,2431.9


From the occupations_en bridge table

In [16]:
# ESCO-Bridge: ISCO-4-digit code -> ESCO-Occupation
esco_isco_bridge = (
    occ_raw
    # occupations only
    .loc[occ_raw["conceptType"] == "Occupation", ["iscoGroup", "conceptUri", "preferredLabel", "altLabels", "description"]]
    .rename(columns={
        "iscoGroup": "isco08_4",
        "preferredLabel": "esco_title_en",
        "altLabels": "esco_alt_labels_en",
        "description": "esco_description_en",
    })
)

# ISCO code: extract only the 4 digits
esco_isco_bridge["isco08_4"] = (
    esco_isco_bridge["isco08_4"]
    .fillna("")
    .astype(str)
    .str.extract(r"(\d{4})", expand=False)
)

# Clean up other text columns
for col in ["esco_title_en", "esco_alt_labels_en", "esco_description_en"]:
    esco_isco_bridge[col] = esco_isco_bridge[col].fillna("").astype(str).str.strip()

# Keep only valid 4-digit ISCO codes
mask_4digit = esco_isco_bridge["isco08_4"].str.len() == 4
esco_isco_bridge = esco_isco_bridge[mask_4digit].dropna(subset=["isco08_4"])

# Remove duplicates
esco_isco_bridge = esco_isco_bridge.drop_duplicates(subset=["isco08_4", "conceptUri"])
print("ESCO-Bridge shape:", esco_isco_bridge.shape)
esco_isco_bridge.head(5)

ESCO-Bridge shape: (3039, 5)


,isco08_4,conceptUri,esco_title_en,esco_alt_labels_en,esco_description_en
0,2654,http://data.europa.eu/esco/occupation/00030d09...,technical director,technical and operations director\nhead of tec...,Technical directors realise the artistic visio...
1,8121,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,metal drawing machine technician\nmetal drawin...,Metal drawing machine operators set up and ope...
2,7543,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,inspector of precision instruments\nprecision ...,Precision device inspectors make sure precisio...
3,3155,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,air traffic safety electronics hardware specia...,Air traffic safety technicians provide technic...
4,2431,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,hospitality revenues manager\nyield manager\nh...,Hospitality revenue managers maximise revenue ...


Join KLDB-ISCO-ESCO

In [17]:
# Link KldB-ISCO to ESCO occupations
kldb_esco = kldb_isco.merge(
    esco_isco_bridge, on="isco08_4",   # shared column
    how="left"
)

print("KldB + ISCO = ESCO Join:", kldb_esco.shape)
kldb_esco.head(5)

KldB + ISCO = ESCO Join: (18391, 12)


,kldb_5_code,kldb_title_de,kldb_title_en,isco08_4,isco_title_de,isco_title_en,unambiguous_flag,focus_and_alt_count,conceptUri,esco_title_en,esco_alt_labels_en,esco_description_en
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,fruit and vegetable picker,fruit picker\nfruit and vegetable harvester\nf...,Fruit and vegetable pickers select and harvest...
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/e5e3ea36...,vineyard worker,grape harvester\ngrape picker\nworker in a vin...,Vineyard workers carry out manual activities r...
2,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9213,Hilfsarbeiter in Ackerbau und Tierhaltung (ohn...,Mixed crop and livestock farm labourers,0,2,http://data.europa.eu/esco/occupation/c9191f7f...,crop production worker,farm worker\nfarm hand\ngrowing worker\ncrop w...,Crop production workers carry out practical ac...
3,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6111,Ackerbauern und Gemüseanbauer,Field crop and vegetable growers,0,2,http://data.europa.eu/esco/occupation/f5e80af5...,agronomic crop production team leader,crop team manager\ncrops production team leade...,Agronomic crop production team leaders are res...
4,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6130,Landwirte mit Ackerbau und Tierhaltung (ohne a...,Mixed crop and animal producers,0,1,http://data.europa.eu/esco/occupation/1c1e86f9...,mixed farmer,crofter\nworking farm manager\nsmallholder,Mixed farmers are responsible for managing all...


Thus: kldb_isco: KldB 5-digit code -> ISCO code (and unique/non-unique flag);
esco_isco_bridge: ISCO code -> ESCO Occupation (occupation_uri, name)

One row in kldb_esco_mapping corresponds to a specific mapping (KldB 5-digit code, ISCO 4-digit code, ESCO Occupation) including title and other attributes.

## 7. Final Mapping Table KldB - ESCO

The join `kldb_esco` results in the mapping table `kldb_esco_mapping`, which contains:
- KldB 5-digit codes + German terms (short/long)
- ISCO-08 4-digit codes + German/English titles
- Flags from the conversion key (unique/alternatives)
- ESCO Occupation (`occupation_uri`) + English title, synonyms, description

The table will later serve as the central basis for KldB base target profiles

In [18]:
# Inspection
print("Spalten in kldb_esco:")
print(kldb_esco.columns.tolist())

# Final Mapping Table
kldb_esco_mapping = pd.DataFrame({
    # KldB Information
    "kldb_5_code": kldb_esco["kldb_5_code"].astype(str).str.strip(),
    "kldb_title_de": kldb_esco["kldb_title_de"].astype(str).str.strip(),
    "kldb_title_en": kldb_esco["kldb_title_en"].astype(str).str.strip(),

    # ISCO Information
    "isco08_4": kldb_esco["isco08_4"].astype(str).str.strip(),
    "isco_title_de": kldb_esco["isco_title_de"].astype(str).str.strip(),
    "isco_title_en": kldb_esco["isco_title_en"].astype(str).str.strip(),

    # Flags from the conversion key
    "unambiguous_flag": kldb_esco["unambiguous_flag"],
    "focus_and_alt_count": kldb_esco["focus_and_alt_count"],

    # ESCO Occupation
    "occupation_uri": kldb_esco["conceptUri"].astype(str).str.strip(),
    "esco_title_en": kldb_esco["esco_title_en"].astype(str).str.strip(),
    "esco_alt_labels_en": kldb_esco["esco_alt_labels_en"].astype(str).str.strip(),
    "esco_description_en": kldb_esco["esco_description_en"].astype(str).str.strip(),
})

# Treat NaN values in text columns as empty strings
text_cols = ["kldb_title_de", "kldb_title_en","isco_title_de", "isco_title_en","occupation_uri", "esco_title_en","esco_alt_labels_en", "esco_description_en",]
for col in text_cols:
    if col in kldb_esco_mapping.columns:
        kldb_esco_mapping[col] = (
            kldb_esco_mapping[col]
            .replace("nan", pd.NA)
            .replace("", pd.NA)
        )

print("kldb_esco_mapping shape:", kldb_esco_mapping.shape)
kldb_esco_mapping.head(5)

Spalten in kldb_esco:
['kldb_5_code', 'kldb_title_de', 'kldb_title_en', 'isco08_4', 'isco_title_de', 'isco_title_en', 'unambiguous_flag', 'focus_and_alt_count', 'conceptUri', 'esco_title_en', 'esco_alt_labels_en', 'esco_description_en']
kldb_esco_mapping shape: (18391, 12)


,kldb_5_code,kldb_title_de,kldb_title_en,isco08_4,isco_title_de,isco_title_en,unambiguous_flag,focus_and_alt_count,occupation_uri,esco_title_en,esco_alt_labels_en,esco_description_en
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,fruit and vegetable picker,fruit picker\nfruit and vegetable harvester\nf...,Fruit and vegetable pickers select and harvest...
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/e5e3ea36...,vineyard worker,grape harvester\ngrape picker\nworker in a vin...,Vineyard workers carry out manual activities r...
2,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9213,Hilfsarbeiter in Ackerbau und Tierhaltung (ohn...,Mixed crop and livestock farm labourers,0,2,http://data.europa.eu/esco/occupation/c9191f7f...,crop production worker,farm worker\nfarm hand\ngrowing worker\ncrop w...,Crop production workers carry out practical ac...
3,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6111,Ackerbauern und Gemüseanbauer,Field crop and vegetable growers,0,2,http://data.europa.eu/esco/occupation/f5e80af5...,agronomic crop production team leader,crop team manager\ncrops production team leade...,Agronomic crop production team leaders are res...
4,11102,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,6130,Landwirte mit Ackerbau und Tierhaltung (ohne a...,Mixed crop and animal producers,0,1,http://data.europa.eu/esco/occupation/1c1e86f9...,mixed farmer,crofter\nworking farm manager\nsmallholder,Mixed farmers are responsible for managing all...


## 8. Quality Check Mapping Table

Number of distinct 5-digit KldB codes in the mapping, KldB codes without an ESCO occupation (`occupation_uri` = NaN), total number of linked ESCO occupations


In [19]:
n_kldb_codes = kldb_esco_mapping["kldb_5_code"].nunique()
n_rows = len(kldb_esco_mapping)
n_missing_esco = kldb_esco_mapping["occupation_uri"].isna().sum()
n_with_esco = n_rows - n_missing_esco

print(f"Anzahl unterschiedlicher KldB-5-Codes: {n_kldb_codes}")
print(f"Anzahl Zeilen gesamt (KldB x ISCO x ESCO): {n_rows}")
print(f"mit ESCO-Occupation (occupation_uri nicht NaN): {n_with_esco}")
print(f"ohne ESCO-Occupation (occupation_uri NaN): {n_missing_esco}")

# Examples without ESCO mapping
print("\nBeispiele ohne ESCO-Occupation (falls vorhanden):")
kldb_esco_mapping[kldb_esco_mapping["occupation_uri"].isna()] \
    .head(10)[["kldb_5_code", "kldb_title_de", "isco08_4", "isco_title_de"]]

Anzahl unterschiedlicher KldB-5-Codes: 1300
Anzahl Zeilen gesamt (KldB x ISCO x ESCO): 18391
mit ESCO-Occupation (occupation_uri nicht NaN): 18378
ohne ESCO-Occupation (occupation_uri NaN): 13

Beispiele ohne ESCO-Occupation (falls vorhanden):


,kldb_5_code,kldb_title_de,isco08_4,isco_title_de
593,12202,Berufe in der Floristik - fachlich ausgerichte...,7549,"Handwerks- und verwandte Berufe, anderweitig n..."
594,12203,Berufe in der Floristik - komplexe Spezialiste...,7549,"Handwerks- und verwandte Berufe, anderweitig n..."
932,21362,Berufe in der Feinoptik - fachlich ausgerichte...,7549,"Handwerks- und verwandte Berufe, anderweitig n..."
14585,81302,Berufe in der Gesundheits- und Krankenpflege (...,3221,Nicht akademische Krankenpflegefachkräfte
14586,81313,Berufe in der Fachkrankenpflege - komplexe Spe...,3221,Nicht akademische Krankenpflegefachkräfte
14587,81323,Berufe in der Fachkinderkrankenpflege - komple...,3221,Nicht akademische Krankenpflegefachkräfte
14589,81333,Berufe in der operations-/medizintechnischen A...,2240,Feldscher und vergleichbare paramedizinische P...
14599,81382,Berufe in der Gesundheits- und Krankenpflege (...,3221,Nicht akademische Krankenpflegefachkräfte
14600,81383,Berufe in der Gesundheits- und Krankenpflege (...,3221,Nicht akademische Krankenpflegefachkräfte
14967,82102,Berufe in der Altenpflege (ohne Spezialisierun...,3221,Nicht akademische Krankenpflegefachkräfte


The remaining (13) KldB-ISCO mappings cannot be mapped to ESCO occupations (occupation_uri is missing). These cases will remain in the mapping.

In [20]:
# wie viele ESCO-Berufe pro KldB-Code gemappt
kldb_per_esco_count = (
    kldb_esco_mapping
    .dropna(subset=["occupation_uri"])
    .groupby("kldb_5_code")["occupation_uri"]
    .nunique()
    .describe()
)
print(kldb_per_esco_count)

count    1287.000000
mean       14.279720
std        13.645515
min         1.000000
25%         4.000000
50%        10.000000
75%        24.000000
max       106.000000
Name: occupation_uri, dtype: float64


## 9. Save KldB - ESCO Mapping

- Result: kldb_esco_mapping (KldB 5-digit × ISCO-08 4-digit × ESCO Occupation) including text attributes + mapping flags.
- Mapping table `kldb_esco_mapping` stored in the `data/interim/` directory

In [21]:
mapping_parquet_path = DATA_INTERIM / "kldb_esco_mapping.parquet"
mapping_csv_path     = DATA_INTERIM / "kldb_esco_mapping.csv"

kldb_esco_mapping.to_parquet(mapping_parquet_path, index=False)
kldb_esco_mapping.to_csv(mapping_csv_path, index=False, encoding="utf-8")

print("KldB <-> ESCO Mapping gespeichert unter:")
print("Parquet:", mapping_parquet_path)
print("CSV:", mapping_csv_path)

KldB <-> ESCO Mapping gespeichert unter:
Parquet: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim\kldb_esco_mapping.parquet
CSV: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim\kldb_esco_mapping.csv


# Summary Notebook 02: KldB Import & Mapping to ESCO

1. Imported the 2020 KldB classification and broken it down into 5-digit codes (occupational categories) (`kldb_5`)
2. Short and long descriptions of KldB counseling positions cleaned up and merged
3. Official conversion keys from KldB 2010 (5-digit) to ISCO-08 (4-digit) imported, cleaned up, and supplemented with flags (unambiguous/ambiguous conversions) (`kldb_isco`)
4. A bridge table from ISCO-08 (4-digit) to ESCO Occupations was created from ESCO Occupations using the `iscoGroup` column (`esco_isco_bridge`)
5. Using a join `KldB-ISCO-ESCO`, the final mapping table `kldb_esco_mapping` was constructed, which contains the linked ISCO-08 codes and ESCO occupations (including English title, synonyms, description) for each KldB 5-digit code. Serves as the base artifact for the following notebooks 03 & 04.
6. Mapping saved in `data/interim/kldb_esco_mapping.parquet` and `data/interim/kldb_esco_mapping.csv`